*메모: 가설1 - 객관적 위치 / 가설2 - 주관적 계층 위치*

# 디지털 사회에서 개인정보 제공 태도는 계층에 따라 달라지는가?

**연구질문**: 경제적 자원이 낮은 사람일수록 할인이나 혜택을 위해 개인정보를 제공할 의향이 높은가?

이 노트북은 KOSSDA `adt_2025_drop_0509.csv` 자료를 사용해 `statsmodels` 기반 회귀분석을 수행한다.

중요한 해석상 주의:
- 이 자료는 관찰자료이므로 회귀계수는 조건부 상관관계로 해석하는 것이 안전하다. 엄밀한 인과효과 주장을 위해서는 무작위 배정, 자연실험, 패널/도구변수 등 추가 식별전략이 필요하다.


## 회귀식

주요 선형 회귀식은 다음과 같다.

$$
Y_i = \beta_0 + \beta_1 Income_i + \beta_2 Rank_i + \beta_3 SatFin_i + \beta_4 FinPros_i + \gamma X_i + \epsilon_i
$$

여기서 종속변수 $Y_i$는 각각 다음 두 문항이다.

- `GIVEPIFR`: 할인이나 무료상품을 받기 위해 개인정보를 제공할 것
- `GIVEPIPC`: 민간기업이 개인정보를 사용하여 이익을 내더라도 제공할 것

핵심 독립변수는 `INCOME` (또는 `INCOM0`), `RANK`, `SATFIN`, `FINPROS`이며, 통제변수 $X_i$에는 `AGE`, `SEX`, `EDUC`, `ABLITINF`, `TECHHARM_R`이 포함된다.  
> 참고로 INCOM0: 월평균 가구소득(원자료/연속형)이고, INCOME: 월평균가구소득(범주형)이다. 

추가로 5점 척도를 이진화한 로짓모형도 추정한다.

$$
Pr(Y_i \ge 4) = logit^{-1}(\beta_0 + \beta_1 Income_i + \beta_2 Rank_i + \beta_3 SatFin_i + \beta_4 FinPros_i + \gamma X_i)
$$

회귀분석과 로짓 분석에 대해 이야기해 보자면

> 1. 선형 회귀분석
>> 5점 척도를 연속형 변수로 처리하여 그대로 사용했다.   
>> 예를 들어 종속변수 GIVEPIFR: 할인이나 무료상품을 받기 위해 개인정보를 제공할 것을 1,2,3,4,5 그대로 두고 회귀분석을 시행했다.

> 2. 로짓분석
>> 1과 같은 종속변수를 이진변수로 바꿔서 분석했다. 



  | 원래 응답값 | 의미 | 이진화 후 |
  |---:|---|---:|
  | 1 | 전혀 그렇지 않다 | 0 |
  | 2 | 그렇지 않다 | 0 |
  | 3 | 보통이다 | 0 |
  | 4 | 그렇다 | 1 |
  | 5 | 매우 그렇다 | 1 
  

즉, GIVEPIFR를 예로 들자면 4 또는 5 를 응답한 사람만 '개인정보 제공 의향 있음'으로 코딩했다. (3은 어디에 포함시킬지 고민하다가... 제공 의향 없음으로 분류했습니다)

이진화한 대상 변수는 다음 두 개입니다.

| 원래 변수 | 이진화 변수 | 의미 |
|---|---|---|
| GIVEPIFR | GIVEPIFR_agree | 할인/무료상품을 받기 위해 개인정보를 제공할 의향 있음 |
| GIVEPIPC | GIVEPIPC_agree | 민간기업이 개인정보로 이익을 내더라도 제공할 의향 있음 |




결론  
  - 선형 회귀분석: 1~5점 점수 자체가 높아지는지 분석
  - 로짓분석: 4~5점, 즉 “동의/의향 있음”일 확률이 높아지는지 분석

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display

In [ ]:
DATA_PATH = Path(r"/home/pck/data_jour/sda_Team Project_/adt_2025_drop_0509.csv")
df_raw = pd.read_csv(DATA_PATH)

print(df_raw.shape)
df_raw.head()


In [ ]:
# 분석에 사용할 변수 전처리
dv_vars = ["GIVEPIFR", "GIVEPIPC"]
core_vars = ["INCOME", "INCOM0", "RANK", "SATFIN", "FINPROS"]
control_vars = ["AGE", "SEX", "EDUC", "ABLITINF", "TECHHARM_R"]
raw_control_vars = ["AGE", "SEX", "EDUC", "ABLITINF", "TECHHARM"]
weight_var = "FINALWT"
analysis_vars_raw = dv_vars + core_vars + raw_control_vars + [weight_var]

missing = [v for v in analysis_vars_raw if v not in df_raw.columns]
if missing:
    raise ValueError(f"데이터에 없는 변수: {missing}")

df_raw[analysis_vars_raw].describe().T


*참고하세요!*

  | 코드상 구분 | 변수명 | 의미 |
  |---|---|---|
  | 종속변수 | GIVEPIFR | 개인정보 제공에 대한 태도1: 할인이나 무료상품을 받기 위해 제공할 것 |
  | 종속변수 | GIVEPIPC | 개인정보 제공에 대한 태도2: 민간기업이 나의 개인정보를 사용하여 이익을 내더라도 제공할 것 |
  | 독립변수 | INCOME | 월평균 가구소득, 범주형 |
  | 독립변수 | INCOM0 | 월평균 가구소득, 원자료/연속형 |
  | 독립변수 | RANK | 주관적인 계층의식, 10점 척도 |
  | 독립변수 | SATFIN | 가계상태 만족도 |
  | 독립변수 | FINPROS | 가계경제상태 전망, 10년 이내 |
  | 통제변수 | AGE | 응답자 연령 |
  | 통제변수 | SEX | 응답자 성별 |
  | 통제변수 | EDUC | 응답자 학력: 최종학교 |
  | 통제변수 | ABLITINF | 인터넷 활용능력3: 인터넷에서 공유하면 안 되는 정보를 구별할 수 있는 것 |
  | 통제변수 | TECHHARM_R | 기술위험 인식 역코딩 변수: 값이 높을수록 기술이 이익보다 해를 더 많이 끼친다고 인식 |
  | 가중치 | FINALWT | 최종 가중치 |

FINALWT 변수에 대해서 설명해 드리자면, 표본가중치 변수입니다. 즉, 설문에 응답한 한 사람이 실제 모집단에서 몇 명을 대표하는지를 반영하는 값입니다.

  예를 들어 설문조사 표본에 어떤 집단이 실제 인구보다 적게 뽑혔다면, 그 집단 응답자에게 더 큰 가중치를 줍니다. 반대로 어떤 집단이 실제
  인구보다 많이 뽑혔다면, 그 집단 응답자에게 더 작은 가중치를 줍니다.


 회귀분석에서는 FINALWT를 사용해서 이렇게 분석했습니다.

  smf.wls(formula=formula, data=model_data, weights=model_data["FINALWT"])

  이건 일반 OLS가 아니라 가중회귀분석 WLS입니다.

  의미는:

  > 가중치가 큰 응답자는 분석에서 더 큰 비중을 갖고, 가중치가 작은 응답자는 더 작은 비중을 갖는다.  
  > 본 분석에서는 표본의 대표성을 보정하기 위해 조사자료에서 제공하는 최종가중치 변수를 적용한 가중회귀분석을 실시한다! (로 쓰면 될 거 같습니다)

## 분석용 데이터 생성

처리 원칙:
- `-8`은 결측으로 처리한다.
- `ABLITINF`의 `-1`은 비해당 값으로 보이므로 결측으로 처리한다.
- `GIVEPI*_agree`: 5점 척도에서 4 또는 5를 동의/의향 있음으로 이진화한 로짓모형용 종속변수이다.


In [ ]:
df = df_raw[analysis_vars_raw].copy()

# 공통 결측 코드 처리
for col in analysis_vars_raw:
    df[col] = df[col].replace(-8, np.nan)

# ABLITINF의 -1은 비해당으로 보이므로 주 모형에서는 결측 처리
df["ABLITINF"] = df["ABLITINF"].replace(-1, np.nan)

# TECHHARM 역코딩: 원점수는 낮을수록 기술위험 인식이 높으므로, 분석에서는 높을수록 위험 인식이 높게 변환
# 원점수 1=매우 동의, 2=동의, 3=중립, 4=반대, 5=매우 반대
# 역코딩 5=매우 동의(위험 인식 높음), 1=매우 반대(위험 인식 낮음)
df["TECHHARM_R"] = 6 - df["TECHHARM"]

analysis_vars = dv_vars + core_vars + control_vars + [weight_var]

# 로짓모형용 이진 종속변수: 4=그렇다, 5=매우 그렇다를 1로 코딩
for y in dv_vars:
    df[f"{y}_agree"] = np.where(df[y].isin([4, 5]), 1,
                         np.where(df[y].isin([1, 2, 3]), 0, np.nan))

# 연속형 소득은 극단값 영향을 줄이기 위해 로그 변환도 생성
df["log_INCOM0"] = np.log1p(df["INCOM0"].where(df["INCOM0"] >= 0, np.nan))

# 표준화 변수: 계수 크기 비교용
## 서로 단위 다른 변수들 비교 가능하도록 같은 척도로 바꾼 변수입니다. 
## 단위가 서로 다른 독립변수들을 평균 0, 표준편차 1로 맞춘 뒤, 어떤 변수가 개인정보 제공 태도와 더 강하게 관련되는지 비교하기 위해서 표준화했습니다~
scale_vars = ["INCOME", "INCOM0", "log_INCOM0", "RANK", "SATFIN", "FINPROS", "AGE", "ABLITINF", "TECHHARM_R"]
for v in scale_vars:
    m = df[v].mean(skipna=True)
    s = df[v].std(skipna=True)
    df[f"z_{v}"] = (df[v] - m) / s

print(df[analysis_vars + ["TECHHARM", "GIVEPIFR_agree", "GIVEPIPC_agree", "log_INCOM0"]].isna().sum())
display(df[["TECHHARM", "TECHHARM_R"]].value_counts(dropna=False).sort_index().rename("count").reset_index())
display(df.head())
display(df.columns)


In [ ]:
# 기술통계
summary_vars = [
    "GIVEPIFR", "GIVEPIPC", "GIVEPIFR_agree", "GIVEPIPC_agree",
    "INCOME", "INCOM0", "log_INCOM0", "RANK", "SATFIN", "FINPROS",
    "AGE", "SEX", "EDUC", "ABLITINF", "TECHHARM", "TECHHARM_R"
]
df[summary_vars].describe().T


In [ ]:
# 상관관계: 핵심 경제자원 변수와 개인정보 제공 태도 (회귀 분석 전 상관관계 확인하는 부분입니다)
import matplotlib.pyplot as plt
import seaborn as sns

corr_vars = ["GIVEPIFR", "GIVEPIPC", "INCOME", "INCOM0", "RANK", "SATFIN", "FINPROS"]
corr_matrix = df[corr_vars].corr(method="pearson")

display(corr_matrix)

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"}
)
plt.title("cor")
plt.tight_layout()
plt.show()

## 회귀분석 기본 가정 점검: 등분산성

회귀분석의 기본 가정 중 하나는 오차항의 분산이 모든 관측치에서 일정하다는 **등분산성**이다. 등분산성이 깨진 경우, 일반 표준오차는 부정확할 수 있으므로 HC3 같은 강건표준오차를 사용하는 것이 적절하다.


해석 기준:
- p-value < 0.05: 등분산성 가정 기각, 이분산성 가능성 있음
- p-value >= 0.05: 등분산성 가정을 기각할 충분한 증거 없음


In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan, het_white

# 최종 주 모형과 동일한 변수 조합으로 예비 OLS를 적합해 등분산성 검정
assumption_formula_income = (
    "{y} ~ INCOME + RANK + SATFIN + FINPROS + AGE "
    "+ C(SEX) + EDUC + ABLITINF + TECHHARM_R"
)

assumption_formula_income_cont = (
    "{y} ~ log_INCOM0 + RANK + SATFIN + FINPROS + AGE "
    "+ C(SEX) + EDUC + ABLITINF + TECHHARM_R"
)

def heteroskedasticity_tests(formula, data, model_name):
    preliminary_ols = smf.ols(formula=formula, data=data).fit()

    bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(
        preliminary_ols.resid,
        preliminary_ols.model.exog
    )
    white_lm, white_lm_p, white_f, white_f_p = het_white(
        preliminary_ols.resid,
        preliminary_ols.model.exog
    )

    return {
        "model": model_name,
        "nobs": int(preliminary_ols.nobs),
        "bp_lm_stat": bp_lm,
        "bp_lm_pvalue": bp_lm_p,
        "bp_f_stat": bp_f,
        "bp_f_pvalue": bp_f_p,
        "white_lm_stat": white_lm,
        "white_lm_pvalue": white_lm_p,
        "white_f_stat": white_f,
        "white_f_pvalue": white_f_p,
        "bp_heteroskedasticity_5pct": bp_lm_p < 0.05,
        "white_heteroskedasticity_5pct": white_lm_p < 0.05,
    }

heteroskedasticity_rows = []
for y in dv_vars:
    heteroskedasticity_rows.append(
        heteroskedasticity_tests(
            assumption_formula_income.format(y=y),
            df,
            f"ASSUMPTION_OLS_{y}_INCOME"
        )
    )
    heteroskedasticity_rows.append(
        heteroskedasticity_tests(
            assumption_formula_income_cont.format(y=y),
            df,
            f"ASSUMPTION_OLS_{y}_logINCOM0"
        )
    )

heteroskedasticity_results = pd.DataFrame(heteroskedasticity_rows)
heteroskedasticity_results


## 주 모형: OLS/WLS, HC3 강건표준오차

5점 척도 종속변수를 연속형으로 보고 선형회귀를 추정한다. 표본가중치 `FINALWT`가 있으므로 WLS를 기본으로 제시하고, HC3 강건표준오차를 사용한다.

`SEX`는 범주형으로, `EDUC`는 순서형 숫자 변수로 처리한다.


In [ ]:
def fit_wls(formula, data, weight_col="FINALWT"):
    model_data = data.dropna(subset=[weight_col]).copy()
    res = smf.wls(formula=formula, data=model_data, weights=model_data[weight_col]).fit(cov_type="HC3")
    return res

def fit_logit(formula, data):
    return smf.logit(formula=formula, data=data).fit(disp=False, maxiter=200, cov_type="HC3")

def tidy_result(res, model_name):
    out = pd.DataFrame({
        "term": res.params.index,
        "coef": res.params.values,
        "std_err": res.bse.values,
        "p_value": res.pvalues.values,
        "ci_low": res.conf_int().iloc[:, 0].values,
        "ci_high": res.conf_int().iloc[:, 1].values,
    })
    out.insert(0, "model", model_name)
    return out

formula_income_cat = (
    "{y} ~ INCOME + RANK + SATFIN + FINPROS + AGE "
    "+ C(SEX) + EDUC + ABLITINF + TECHHARM_R"
)

formula_income_cont = (
    "{y} ~ log_INCOM0 + RANK + SATFIN + FINPROS + AGE "
    "+ C(SEX) + EDUC + ABLITINF + TECHHARM_R"
)

models = {}
for y in dv_vars:
    models[f"OLS_WLS_{y}_INCOME"] = fit_wls(formula_income_cat.format(y=y), df)
    models[f"OLS_WLS_{y}_logINCOM0"] = fit_wls(formula_income_cont.format(y=y), df)

key_terms = ["INCOME", "log_INCOM0", "RANK", "SATFIN", "FINPROS", "ABLITINF", "TECHHARM_R"]
tidy_ols = pd.concat([tidy_result(res, name) for name, res in models.items()], ignore_index=True)
tidy_ols[tidy_ols["term"].isin(key_terms)].sort_values(["model", "term"])


In [ ]:
# 전체 회귀표
for name, res in models.items():
    print("=" * 100)
    print(name)
    print(res.summary())


In [ ]:
# 통계적으로 유의미한 관계만 표시: p < 0.05
significant_ols = tidy_ols[tidy_ols["p_value"] < 0.05].copy()

# 절편은 제외
significant_ols = significant_ols[significant_ols["term"] != "Intercept"]

significant_ols = significant_ols.sort_values(["model", "p_value"])
significant_ols[["model", "term", "coef", "std_err", "p_value", "ci_low", "ci_high"]]


### 해석할 때 주의!

소득, 나이, 성별, 학력, 정보능력(ABLITINF), 미래재망(FINPROS) 등 여러 변수가 한꺼번에 포함되어 있는데, ABLITINF를 기준으로 본다는 것은 만약 두 사람의 소득, 나이, 성별, 학력, 계층인식, 심지어 미래 재정 전망(FINPROS)까지 완벽하게 똑같다고 가정할 때, 오직 '정보 활용 능력(ABLITINF)'만 1단위 다르면 기부(GIVEPIFR)에 어떤 차이가 생길까?"를 묻는 것이다. 

즉 나머지 변수를 통제하고 term의 효과를 보겠다 이런식으로 해석해야 함~

## 표준화 계수 모형

핵심 독립변수의 상대적 크기를 비교하기 위해 연속형 변수들을 z-score로 표준화한 모형을 추정한다. 종속변수는 원래 1-5 척도를 유지한다.


In [ ]:
formula_z = (
    "{y} ~ z_INCOME + z_RANK + z_SATFIN + z_FINPROS + z_AGE "
    "+ C(SEX) + EDUC + z_ABLITINF + z_TECHHARM_R"
)

z_models = {f"Z_WLS_{y}": fit_wls(formula_z.format(y=y), df) for y in dv_vars}
tidy_z = pd.concat([tidy_result(res, name) for name, res in z_models.items()], ignore_index=True)
z_key_terms = ["z_INCOME", "z_RANK", "z_SATFIN", "z_FINPROS", "z_AGE", "z_ABLITINF", "z_TECHHARM_R"]
tidy_z[tidy_z["term"].isin(z_key_terms)].sort_values(["model", "term"])


## 로짓모형: 개인정보 제공 의향 있음 여부

종속변수를 `4` 또는 `5` 응답이면 1, `1`~`3`이면 0으로 이진화해 로짓모형을 추정한다. 계수는 로그오즈이므로, 평균 한계효과도 함께 계산한다.


In [ ]:
logit_formula = (
    "{y}_agree ~ INCOME + RANK + SATFIN + FINPROS + AGE "
    "+ C(SEX) + EDUC + ABLITINF + TECHHARM_R"
)

logit_models = {}
for y in dv_vars:
    logit_models[f"LOGIT_{y}_agree"] = fit_logit(logit_formula.format(y=y), df)

tidy_logit = pd.concat([tidy_result(res, name) for name, res in logit_models.items()], ignore_index=True)
tidy_logit[tidy_logit["term"].isin(key_terms)].sort_values(["model", "term"])


In [ ]:
# 평균 한계효과(average marginal effects)
## 독립변수가 1단위 증가할 때, 종속변수가 1이 될 확률이 평균적으로 얼마나 변하는가?로 해석할 수 있다. 
for name, res in logit_models.items():
    print("=" * 100)
    print(name)
    margeff = res.get_margeff(at="overall")
    print(margeff.summary())
    

In [ ]:
# 로짓모형에서 통계적으로 유의미한 관계만 표시: p < 0.05
significant_logit = tidy_logit[tidy_logit["p_value"] < 0.05].copy()

# 절편은 연구질문 해석 대상이 아니므로 제외
significant_logit = significant_logit[significant_logit["term"] != "Intercept"]

significant_logit = significant_logit.sort_values(["model", "p_value"])
significant_logit[["model", "term", "coef", "std_err", "p_value", "ci_low", "ci_high"]]


## 결과 저장

핵심 회귀계수 표를 CSV로 저장한다.

> 필요하면 주석 풀고 다운 받으세요!!


In [ ]:
# OUT_DIR = DATA_PATH.parent

# result_tables = {
#     "privacy_heteroskedasticity_tests.csv": heteroskedasticity_results,
#     "privacy_wls_core_coefficients.csv": tidy_ols,
#     "privacy_wls_standardized_coefficients.csv": tidy_z,
#     "privacy_logit_coefficients.csv": tidy_logit,
# }

# for filename, table in result_tables.items():
#     path = OUT_DIR / filename
#     table.to_csv(path, index=False, encoding="utf-8-sig")
#     print(path)


## 해석 가이드

- 연구질문에 대한 핵심 검정은 `INCOME` 또는 `log_INCOM0` 계수다.
- `INCOME`/`log_INCOM0` 계수가 음수이고 통계적으로 유의하면, 소득이 낮을수록 개인정보 제공 의향이 높다는 가설과 부합한다.
- `RANK`, `SATFIN`, `FINPROS`도 음수이면 주관적 계층/경제상태가 낮을수록 개인정보 제공 의향이 높다는 방향이다.

## 축약 통제변수 모형: AGE, SEX, EDUC만 포함

아래 모형은 요청한 대로 통제변수를 `control_vars = ["AGE", "SEX", "EDUC"]`로 제한한 버전이다. 핵심 독립변수(`INCOME` 또는 `log_INCOM0`, `RANK`, `SATFIN`, `FINPROS`)는 유지하고, 통제변수는 연령, 성별, 교육수준만 포함한다.

회귀식은 다음과 같다.

$$
Y_i = \beta_0 + \beta_1 Income_i + \beta_2 Rank_i + \beta_3 SatFin_i + \beta_4 FinPros_i + \delta_1 Age_i + \delta_2 Sex_i + \delta_3 Educ_i + \epsilon_i
$$


In [ ]:
# 축약 통제변수: AGE, SEX, EDUC만 사용
control_vars_basic = ["AGE", "SEX", "EDUC"]

formula_basic_income = (
    "{y} ~ INCOME + RANK + SATFIN + FINPROS + AGE "
    "+ C(SEX) + EDUC"
)

formula_basic_income_cont = (
    "{y} ~ log_INCOM0 + RANK + SATFIN + FINPROS + AGE "
    "+ C(SEX) + EDUC"
)

basic_models = {}
for y in dv_vars:
    basic_models[f"BASIC_WLS_{y}_INCOME"] = fit_wls(formula_basic_income.format(y=y), df)
    basic_models[f"BASIC_WLS_{y}_logINCOM0"] = fit_wls(formula_basic_income_cont.format(y=y), df)

tidy_basic = pd.concat([tidy_result(res, name) for name, res in basic_models.items()], ignore_index=True)
basic_key_terms = ["INCOME", "log_INCOM0", "RANK", "SATFIN", "FINPROS", "AGE", "C(SEX)[T.2.0]", "EDUC"]
tidy_basic[tidy_basic["term"].isin(basic_key_terms)].sort_values(["model", "term"])


In [ ]:
# 축약 통제변수 로짓모형: 개인정보 제공 의향 있음(4 또는 5) 여부
logit_formula_basic = (
    "{y}_agree ~ INCOME + RANK + SATFIN + FINPROS + AGE "
    "+ C(SEX) + EDUC"
)

basic_logit_models = {}
for y in dv_vars:
    basic_logit_models[f"BASIC_LOGIT_{y}_agree"] = fit_logit(logit_formula_basic.format(y=y), df)

tidy_basic_logit = pd.concat([tidy_result(res, name) for name, res in basic_logit_models.items()], ignore_index=True)
tidy_basic_logit[tidy_basic_logit["term"].isin(basic_key_terms)].sort_values(["model", "term"])


In [ ]:
# 축약 통제변수 로짓모형의 평균 한계효과
for name, res in basic_logit_models.items():
    print("=" * 100)
    print(name)
    margeff = res.get_margeff(at="overall") 
    print(margeff.summary())


In [ ]:
# 축약 통제변수 로짓모형에서 통계적으로 유의미한 관계만 표시: p < 0.05
significant_basic_logit = tidy_basic_logit[tidy_basic_logit["p_value"] < 0.05].copy()

# 절편은 연구질문 해석 대상이 아니므로 제외
significant_basic_logit = significant_basic_logit[significant_basic_logit["term"] != "Intercept"]

significant_basic_logit = significant_basic_logit.sort_values(["model", "p_value"])
significant_basic_logit[["model", "term", "coef", "std_err", "p_value", "ci_low", "ci_high"]]


In [ ]:
# # 축약 통제변수 모형 결과 저장
# basic_wls_path = OUT_DIR / "privacy_basic_controls_wls_coefficients.csv"
# basic_logit_path = OUT_DIR / "privacy_basic_controls_logit_coefficients.csv"

# tidy_basic.to_csv(basic_wls_path, index=False, encoding="utf-8-sig")
# tidy_basic_logit.to_csv(basic_logit_path, index=False, encoding="utf-8-sig")

# print(basic_wls_path)
# print(basic_logit_path)


## 한국어 회귀식

축약 통제변수 모형의 회귀식은 한국어로 다음과 같이 표현할 수 있다.

$$
개인정보제공태도_i
=
\beta_0
+
\beta_1 가구소득_i
+
\beta_2 주관적계층의식_i
+
\beta_3 가계상태만족도_i
+
\beta_4 가계경제전망_i
+
\delta_1 연령_i
+
\delta_2 성별_i
+
\delta_3 교육수준_i
+
\epsilon_i
$$

문장으로 풀어 쓰면, 개인 $i$의 개인정보 제공 태도는 가구소득, 주관적 계층의식, 가계상태 만족도, 가계경제 전망의 영향을 받으며, 연령, 성별, 교육수준을 통제한 회귀모형으로 추정한다.

여기서 `개인정보제공태도`는 분석에 따라 `GIVEPIFR` 또는 `GIVEPIPC`를 의미한다.


## 독립변수 다중공선성 점검: VIF

다중공선성은 종속변수와 독립변수의 관계가 아니라 **독립변수들끼리 강하게 관련되어 있는지**의 문제다. 따라서 실제 회귀식에 동시에 들어가는 독립변수들을 대상으로 VIF(Variance Inflation Factor)를 계산한다.

해석 기준은 일반적으로 다음과 같다.

- VIF 1: 공선성 거의 없음
- VIF 2~5: 어느 정도 관련은 있으나 대체로 허용 가능
- VIF 5 이상: 주의 필요
- VIF 10 이상: 심각한 다중공선성 가능성

`INCOME`과 `INCOM0`/`log_INCOM0`는 서로 상관이 높을 수 있으므로 같은 모형에 동시에 넣지 않고, 아래처럼 `INCOME` 모형과 `log_INCOM0` 모형을 분리해서 VIF를 확인한다.


In [ ]:
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 실제 주 회귀식과 같은 독립변수 조합으로 VIF 계산
vif_formula_income = (
    "GIVEPIFR ~ INCOME + RANK + SATFIN + FINPROS + AGE "
    "+ C(SEX) + EDUC + ABLITINF + TECHHARM_R"
)

vif_formula_log_income = (
    "GIVEPIFR ~ log_INCOM0 + RANK + SATFIN + FINPROS + AGE "
    "+ C(SEX) + EDUC + ABLITINF + TECHHARM_R"
)

def calculate_vif(formula, data, model_name):
    # dmatrices를 쓰면 C(SEX) 같은 formula 항을 회귀식과 동일하게 설계행렬로 변환할 수 있다.
    y, X = dmatrices(formula, data=data, return_type="dataframe")

    rows = []
    for i, col in enumerate(X.columns):
        if col == "Intercept":
            continue
        rows.append({
            "model": model_name,
            "term": col,
            "vif": variance_inflation_factor(X.values, i),
        })
    out = pd.DataFrame(rows).sort_values("vif", ascending=False).reset_index(drop=True)
    return out

vif_income = calculate_vif(vif_formula_income, df, "VIF_INCOME_model")
vif_log_income = calculate_vif(vif_formula_log_income, df, "VIF_logINCOM0_model")

vif_results = pd.concat([vif_income, vif_log_income], ignore_index=True)
vif_results
# 해보니까 문제 없네요~

In [ ]:
# # VIF 결과 저장
# vif_path = OUT_DIR / "privacy_vif_multicollinearity.csv"
# vif_results.to_csv(vif_path, index=False, encoding="utf-8-sig")
# print(vif_path)


## 교수님께 질문하고 싶은 부분 & 논의가 필요한 부분

1. 회귀 모형(+ 가중회귀 모형)과 로짓 모형 중에 어떤 방법론을 선택하는게 타당할까(각 방법의 이유가 있을거 같은데...)

2. 상관관계는 매우 낮은데, 회귀 계수가 유의미하게 나온 상황에서 어떻게 해석해야 하는가? 

3. 강건표준오차 HC3 사용에 관해서. HC3 사용이 타당한가?

4. income 과 logincom0 변수를 어떻게 처리할지.. 따로따로? 아니면 둘 다?
> INCOM0: 월평균 가구소득(원료/연속형)
> INCOME: 월평균가구소득(범주형)

즉, 가구소득으로 했음. 개인소득으로 해야할까..?

5. 종속변수 두 개? -> 내가 봤을 때는 GIVEPIFR: 개인정보제공에대한태도1:할인이나무료상품을받기위해제공할것  
GIVEPIPC: 개인정보제공에대한태도2:민간기업이나의개인정보를사용하여이익을내더라도제공할것  

중에서 GIVEPIFR는 무조건인데...
> 왜냐하면 우리가 보고 싶은 것은 혜택의 의미이기 떄문에 민간기업이 나의 개인정보로 이익을 내더라도 줄 것은 나에게 곧바로 혜택이 돌아오는 차원의 이야기는 아니니까. 종속변수 하나로 가야하맂 않을까?